# Creates table S1

In [ ]:
import re
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import matthews_corrcoef, confusion_matrix
from scipy.stats import fisher_exact

In [ ]:
path = ("../Results/CRISPR_final_results.csv")

In [ ]:
table = pd.read_csv(path)

## Summarize data (ALL data, NOT per feature)

In [ ]:
def summarize_all_data_cutoffs(data):
    cell_lines = ['HepG2', 'K562']
    shap_cutoffs = [0, 0.05, 0.1, 0.2]
    rows = []
    for cell_line in cell_lines:
        for cutoff in shap_cutoffs:
            # ------------------------------------------------------------------
            # 1. Subset & significance filter
            # ------------------------------------------------------------------
            subset = data[data['cell_line'] == cell_line].copy()
            subset['Significance'] = (
                (subset['dPSI'].abs() > 0) &
                (subset['FDR'] <= 0.1) &
                (subset['CTRL_SHAP'].abs() >= cutoff)
            ).astype(int)
            significant_subset = subset[subset['Significance'] == 1].copy()
            num_significant = len(significant_subset)
            
            # ------------------------------------------------------------------
            # 2. MCC
            # ------------------------------------------------------------------
            y_true = np.sign(significant_subset['dPSI'].values).astype(int)
            y_pred = np.sign(significant_subset['CTRL_SHAP'].values).astype(int)
            
            nonzero_mask = (y_true != 0) & (y_pred != 0)
            y_true_nz = y_true[nonzero_mask]
            y_pred_nz = y_pred[nonzero_mask]

            
            mcc = matthews_corrcoef(y_true_nz, y_pred_nz)
            tp = ((y_pred_nz ==  1) & (y_true_nz ==  1)).sum()
            fp = ((y_pred_nz ==  1) & (y_true_nz == -1)).sum()
            fn = ((y_pred_nz == -1) & (y_true_nz ==  1)).sum()
            tn = ((y_pred_nz == -1) & (y_true_nz == -1)).sum()
            
            # ------------------------------------------------------------------
            # 3. dPSI–SHAP concordance
            # ------------------------------------------------------------------
            concordance = np.mean(y_true_nz == y_pred_nz) * 100
            rows.append({
                'Cell Line': cell_line,
                'Local SHAP Cutoff': cutoff,
                '# rMATS Significant Events': num_significant,
                '# True Positives': tp, 
                '# False Positives': fp, 
                '# True Negatives': tn, 
                '# False Negatives': fn,
                '% Concordant': concordance,
                'MCC': mcc
            })
    return pd.DataFrame(rows).sort_values(['Cell Line', 'Local SHAP Cutoff']).reset_index(drop=True)

In [ ]:
all_data_cut = summarize_all_data_cutoffs(table)

In [ ]:
all_data_cut.to_csv("CRISPR_dPSI_vs_CTRL_SHAP_MCC.tsv", index=False, sep="\t")